# Transformation des fichiers Excel historiques vers les tables POLYPBASE

Ce notebook lit les fichiers `Suivi_*.xlsx`, nettoie les données historiques, détecte les anomalies, puis prépare les tables compatibles avec le modèle de données POLYPBASE.

Tables produites :
- `espece`
- `souche`
- `boite`
- `saisir_releve`
- `zone_thermique`
- `range`

Les fichiers Excel sources, les exports CSV et les anomalies ne doivent pas être versionnés sur GitHub.


## 1. Configuration et chargement des fichiers


In [ ]:
from pathlib import Path
import re

import numpy as np
import pandas as pd

# Dossier du projet.
# Si le notebook est lancé depuis /notebooks, on remonte à la racine du projet.
PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent

# Par défaut, les fichiers Excel sont attendus dans un dossier data/ non versionné.
# Si besoin, adapte uniquement cette ligne.
DATA_DIR = PROJECT_DIR / "data"

# Dossier de sortie pour les tables propres et les anomalies.
OUTPUT_DIR = PROJECT_DIR / "exports_polypbase"
ANOMALY_DIR = OUTPUT_DIR / "anomalies"

OUTPUT_DIR.mkdir(exist_ok=True)
ANOMALY_DIR.mkdir(parents=True, exist_ok=True)

excel_files = sorted(DATA_DIR.glob("Suivi_*.xlsx"))

# Sécurité : si aucun fichier n'est trouvé dans data/, on cherche à la racine du projet.
if not excel_files:
    excel_files = sorted(PROJECT_DIR.glob("Suivi_*.xlsx"))

print("Dossier projet :", PROJECT_DIR)
print("Dossier données :", DATA_DIR)
print("Nombre de fichiers Excel trouvés :", len(excel_files))
display([file.name for file in excel_files])


In [ ]:
# Vérification rapide des feuilles disponibles dans chaque fichier.

for file in excel_files:
    xls = pd.ExcelFile(file)
    print(f"\n{file.name}")
    print(xls.sheet_names)


## 2. Fonctions de lecture et de normalisation

Les fichiers Excel historiques ont une structure particulière :
- les semaines sont en ligne 2 ;
- les colonnes principales sont l'espèce, le code boîte, la température et le type de mesure ;
- les valeurs biologiques sont réparties par semaine ;
- certaines cellules sont vides car elles sont visuellement fusionnées dans Excel.


In [ ]:
def extract_year_from_filename(path: Path) -> int | None:
    # Extrait l'année depuis un nom de fichier du type Suivi_2025.xlsx.
    match = re.search(r"(\d{4})", path.name)
    return int(match.group(1)) if match else None


def clean_text(value):
    # Nettoie une cellule texte et retourne None si elle est vide.
    if pd.isna(value):
        return None
    value = str(value).strip()
    return value if value else None


def normalize_measure_type(value):
    # Normalise le type de mesure biologique.
    value = clean_text(value)
    if value is None:
        return None

    value_lower = value.lower()

    if "polype" in value_lower:
        return "polypes"
    if "éphyr" in value_lower or "ephyr" in value_lower:
        return "ephyrules"

    return value_lower


def parse_excel_file(file_path: Path) -> pd.DataFrame:
    # Transforme un fichier Excel annuel en format long.
    year = extract_year_from_filename(file_path)
    xls = pd.ExcelFile(file_path)

    rows = []

    for sheet_name in xls.sheet_names:
        # Les feuilles Feuil1 / Feuil2 sont génériques ou vides.
        if sheet_name.lower().startswith("feuil"):
            continue

        df = pd.read_excel(file_path, sheet_name=sheet_name, header=None)

        # Les semaines sont sur la ligne 1, à partir de la colonne 4.
        week_columns = []
        for col in range(4, df.shape[1]):
            week_value = df.iloc[1, col]

            if pd.notna(week_value):
                try:
                    week_number = int(float(week_value))
                    week_columns.append((col, week_number))
                except ValueError:
                    pass

        current_species = None
        current_box = None
        current_temperature = None

        # Les données commencent à la ligne 2.
        for row_idx in range(2, df.shape[0]):
            species = clean_text(df.iloc[row_idx, 0])
            box = clean_text(df.iloc[row_idx, 1])
            temperature = df.iloc[row_idx, 2]
            measure_type = normalize_measure_type(df.iloc[row_idx, 3])

            # Si une nouvelle ligne de boîte/espèce commence, on met à jour le contexte.
            # Si la température est vide au début d'un nouveau bloc, elle reste manquante.
            new_block = species is not None or box is not None

            if species is not None:
                current_species = species

            if box is not None:
                current_box = box

            if new_block:
                current_temperature = temperature if pd.notna(temperature) else None
            elif pd.notna(temperature):
                current_temperature = temperature

            if measure_type is None:
                continue

            for col, week_number in week_columns:
                rows.append({
                    "annee": year,
                    "groupe": sheet_name,
                    "espece": current_species,
                    "code_boite": current_box,
                    "temperature_cible": current_temperature,
                    "semaine": week_number,
                    "type_mesure": measure_type,
                    "valeur": df.iloc[row_idx, col],
                    "fichier_source": file_path.name,
                    "feuille_source": sheet_name,
                    "ligne_source": row_idx + 1,
                })

    return pd.DataFrame(rows)


## 3. Lecture de tous les fichiers Excel


In [ ]:
tracking_long_df = pd.concat(
    [parse_excel_file(file) for file in excel_files],
    ignore_index=True
)

display(tracking_long_df.head(20))
print("Dimensions :", tracking_long_df.shape)
print(tracking_long_df["type_mesure"].value_counts(dropna=False))


## 4. Cas particulier Staurozoa

Dans la feuille `Staurozoa`, certaines lignes sont notées `Nb`.
Après vérification métier, les lignes fonctionnent par paire :
- premier `Nb` = polypes ;
- second `Nb` = éphyrules.


In [ ]:
# Export de contrôle des lignes Staurozoa avant correction.
stauro_values_df = tracking_long_df[
    (tracking_long_df["groupe"] == "Staurozoa")
    & (tracking_long_df["valeur"].notna())
].copy()

stauro_values_df.to_csv(
    ANOMALY_DIR / "staurozoa_a_verifier.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Lignes Staurozoa à vérifier :", stauro_values_df.shape)
display(stauro_values_df.head(20))


In [ ]:
# Correction spécifique pour Staurozoa.

tracking_long_df = tracking_long_df.sort_values(
    ["fichier_source", "feuille_source", "ligne_source", "semaine"]
).copy()

stauro_mask = (
    (tracking_long_df["groupe"] == "Staurozoa")
    & (tracking_long_df["type_mesure"] == "nb")
)

stauro_lignes = (
    tracking_long_df[stauro_mask][["fichier_source", "feuille_source", "ligne_source"]]
    .drop_duplicates()
    .sort_values(["fichier_source", "feuille_source", "ligne_source"])
    .reset_index(drop=True)
)

stauro_lignes["type_corrige"] = np.where(
    stauro_lignes.index % 2 == 0,
    "polypes",
    "ephyrules"
)

tracking_long_df = tracking_long_df.merge(
    stauro_lignes,
    on=["fichier_source", "feuille_source", "ligne_source"],
    how="left"
)

tracking_long_df["type_mesure"] = tracking_long_df["type_corrige"].fillna(
    tracking_long_df["type_mesure"]
)

tracking_long_df = tracking_long_df.drop(columns=["type_corrige"])

print(tracking_long_df["type_mesure"].value_counts(dropna=False))


## 5. Nettoyage général des relevés


In [ ]:
clean_tracking_df = tracking_long_df.copy()

# Garder uniquement les lignes avec une valeur renseignée.
clean_tracking_df = clean_tracking_df[clean_tracking_df["valeur"].notna()].copy()

# Garder uniquement les mesures biologiques exploitables.
clean_tracking_df = clean_tracking_df[
    clean_tracking_df["type_mesure"].isin(["polypes", "ephyrules"])
].copy()

# Nettoyer les textes.
for col in ["groupe", "espece", "code_boite", "fichier_source", "feuille_source"]:
    clean_tracking_df[col] = clean_tracking_df[col].astype("string").str.strip()

clean_tracking_df["code_boite"] = (
    clean_tracking_df["code_boite"]
    .str.replace("\n", "", regex=False)
    .str.replace("\r", "", regex=False)
    .str.strip()
)

clean_tracking_df["valeur"] = pd.to_numeric(clean_tracking_df["valeur"], errors="coerce")
clean_tracking_df["temperature_cible"] = pd.to_numeric(
    clean_tracking_df["temperature_cible"],
    errors="coerce"
)

clean_tracking_df = clean_tracking_df[clean_tracking_df["valeur"].notna()].copy()

print("Données propres :", clean_tracking_df.shape)
print(clean_tracking_df["type_mesure"].value_counts(dropna=False))
print("Espèces manquantes :", clean_tracking_df["espece"].isna().sum())
print("Boîtes manquantes :", clean_tracking_df["code_boite"].isna().sum())
print("Températures manquantes :", clean_tracking_df["temperature_cible"].isna().sum())


## 6. Extraction des codes métier

Exemple de code boîte : `ALA-JKA-1.02`

- `ALA` : code local de l'espèce ;
- `JKA` : code de provenance ;
- `1` : numéro local de souche ;
- `02` : numéro local de boîte ;
- `ALA-JKA-1` : code de souche reconstitué.


In [ ]:
def extract_code_espece_local(code_boite):
    if pd.isna(code_boite):
        return None
    parts = str(code_boite).strip().split("-")
    return parts[0] if len(parts) >= 1 else None


def extract_provenance(code_boite):
    if pd.isna(code_boite):
        return None
    parts = str(code_boite).strip().split("-")
    return parts[1] if len(parts) >= 2 else None


def extract_numero_souche_local(code_boite):
    if pd.isna(code_boite):
        return None
    parts = str(code_boite).strip().split("-")
    if len(parts) >= 3:
        return parts[2].split(".")[0]
    return None


def extract_numero_boite_local(code_boite):
    if pd.isna(code_boite):
        return None
    code_boite = str(code_boite).strip()
    return code_boite.split(".")[-1] if "." in code_boite else None


def extract_code_souche(code_boite):
    if pd.isna(code_boite):
        return None
    code_boite = str(code_boite).strip()
    return code_boite.split(".")[0] if "." in code_boite else code_boite


clean_tracking_df["code_espece_local"] = clean_tracking_df["code_boite"].apply(extract_code_espece_local)
clean_tracking_df["code_souche"] = clean_tracking_df["code_boite"].apply(extract_code_souche)
clean_tracking_df["numero_souche_local"] = clean_tracking_df["code_boite"].apply(extract_numero_souche_local)
clean_tracking_df["code_provenance"] = clean_tracking_df["code_boite"].apply(extract_provenance)
clean_tracking_df["numero_boite_local"] = clean_tracking_df["code_boite"].apply(extract_numero_boite_local)

display(clean_tracking_df[[
    "code_boite",
    "code_espece_local",
    "code_souche",
    "numero_souche_local",
    "code_provenance",
    "numero_boite_local",
    "espece"
]].drop_duplicates().head(30))


## 7. Corrections des noms d'espèces

Les corrections automatiques ne concernent que les fautes ou variantes évidentes.
Les cas ambigus sont exportés dans les fichiers d'anomalies.


In [ ]:
species_corrections = {
    "Alatina morandini": "Alatina morandinii",
    "Tiarropsis multicirrata": "Tiaropsis multicirrata",
    "Chrysaoa achlyos": "Chrysaora achlyos",
    "Cyanea lamarcki": "Cyanea lamarckii",
    "Clytia hemispherica": "Clytia hemisphaerica",
    "Cyanea nozakii Kishinouye": "Cyanea nozakii",
    "eudendryum sp": "Eudendrium sp.",
}

# Garder une trace des corrections effectuées.
species_corrections_df = pd.DataFrame([
    {"nom_initial": initial, "nom_corrige": corrected}
    for initial, corrected in species_corrections.items()
])

species_corrections_df.to_csv(
    ANOMALY_DIR / "corrections_noms_especes.csv",
    index=False,
    encoding="utf-8-sig"
)

clean_tracking_df["espece"] = clean_tracking_df["espece"].replace(species_corrections)

display(species_corrections_df)


## 8. Détection des anomalies


In [ ]:
# Anomalie 1 : températures manquantes.
anomalies_temperature_df = clean_tracking_df[
    clean_tracking_df["temperature_cible"].isna()
].copy()

anomalies_temperature_df.to_csv(
    ANOMALY_DIR / "anomalies_temperature_manquante.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Lignes avec température manquante :", len(anomalies_temperature_df))

display(
    anomalies_temperature_df[[
        "fichier_source",
        "feuille_source",
        "code_boite",
        "espece",
        "annee",
        "semaine"
    ]]
    .drop_duplicates()
    .sort_values(["fichier_source", "feuille_source", "code_boite", "semaine"])
    .head(30)
)


In [ ]:
# Anomalie 2 : même code de souche associé à plusieurs espèces.
species_per_souche = (
    clean_tracking_df[["code_souche", "espece"]]
    .drop_duplicates()
    .groupby("code_souche")["espece"]
    .nunique()
    .reset_index(name="nb_especes")
    .sort_values("nb_especes", ascending=False)
)

problem_souches = species_per_souche[
    species_per_souche["nb_especes"] > 1
]["code_souche"].tolist()

anomalies_souche_df = (
    clean_tracking_df[
        clean_tracking_df["code_souche"].isin(problem_souches)
    ][[
        "code_souche",
        "code_boite",
        "espece",
        "fichier_source",
        "feuille_source"
    ]]
    .drop_duplicates()
    .sort_values(["code_souche", "espece", "code_boite"])
)

anomalies_souche_df.to_csv(
    ANOMALY_DIR / "anomalies_souche_plusieurs_especes.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Codes de souche associés à plusieurs espèces :", len(problem_souches))
display(species_per_souche[species_per_souche["nb_especes"] > 1])
display(anomalies_souche_df.head(50))


In [ ]:
# Anomalie 3 : même code de boîte associé à plusieurs espèces.
species_per_boite = (
    clean_tracking_df[["code_boite", "espece"]]
    .drop_duplicates()
    .groupby("code_boite")["espece"]
    .nunique()
    .reset_index(name="nb_especes")
    .sort_values("nb_especes", ascending=False)
)

codes_boites_ambigues = species_per_boite[
    species_per_boite["nb_especes"] > 1
]["code_boite"].tolist()

anomalies_boites_df = (
    clean_tracking_df[
        clean_tracking_df["code_boite"].isin(codes_boites_ambigues)
    ][[
        "code_boite",
        "code_souche",
        "espece",
        "fichier_source",
        "feuille_source"
    ]]
    .drop_duplicates()
    .sort_values(["code_boite", "espece", "fichier_source"])
)

anomalies_boites_df.to_csv(
    ANOMALY_DIR / "anomalies_boites_plusieurs_especes.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Codes de boîte associés à plusieurs espèces :", len(codes_boites_ambigues))
display(species_per_boite[species_per_boite["nb_especes"] > 1])
display(anomalies_boites_df.head(50))


In [ ]:
# Données valides pour les tables finales :
# on exclut les boîtes qui changent d'espèce selon les années.
clean_tracking_valid_df = clean_tracking_df[
    ~clean_tracking_df["code_boite"].isin(codes_boites_ambigues)
].copy()

clean_tracking_anomalies_df = clean_tracking_df[
    clean_tracking_df["code_boite"].isin(codes_boites_ambigues)
].copy()

print("Lignes valides :", len(clean_tracking_valid_df))
print("Lignes mises en anomalie :", len(clean_tracking_anomalies_df))
print("Boîtes ambiguës :", len(codes_boites_ambigues))


## 9. Création des tables principales


In [ ]:
# Table ESPECE, à partir des données valides.

espece_df = (
    clean_tracking_valid_df[["espece"]]
    .dropna()
    .drop_duplicates()
    .sort_values("espece")
    .reset_index(drop=True)
)

espece_df["id_espece"] = range(1, len(espece_df) + 1)

espece_df["code_espece"] = (
    espece_df["espece"]
    .str.upper()
    .str.replace(" ", "_", regex=False)
    .str.replace(".", "", regex=False)
)

espece_df = espece_df.rename(columns={"espece": "nom_scientifique"})

espece_df = espece_df[[
    "id_espece",
    "code_espece",
    "nom_scientifique"
]]

print("Nombre d'espèces :", len(espece_df))
display(espece_df.head(20))


In [ ]:
# Table SOUCHE, à partir des données valides.

souche_df = (
    clean_tracking_valid_df[[
        "code_souche",
        "numero_souche_local",
        "espece"
    ]]
    .dropna(subset=["code_souche", "espece"])
    .drop_duplicates()
    .reset_index(drop=True)
)

souche_df = souche_df.merge(
    espece_df[["id_espece", "nom_scientifique"]],
    left_on="espece",
    right_on="nom_scientifique",
    how="left"
)

souche_df["id_souche"] = range(1, len(souche_df) + 1)
souche_df["nom_souche"] = souche_df["code_souche"]

souche_df = souche_df[[
    "id_souche",
    "code_souche",
    "numero_souche_local",
    "nom_souche",
    "id_espece"
]]

print("Nombre de souches :", len(souche_df))
print("Souches sans espèce liée :", souche_df["id_espece"].isna().sum())
display(souche_df.head(20))


In [ ]:
# Table BOITE, à partir des données valides.

boite_df = (
    clean_tracking_valid_df[[
        "code_boite",
        "code_souche",
        "numero_boite_local",
        "espece"
    ]]
    .dropna(subset=["code_boite", "code_souche", "espece"])
    .drop_duplicates()
    .reset_index(drop=True)
)

boite_df = boite_df.merge(
    espece_df[["id_espece", "nom_scientifique"]],
    left_on="espece",
    right_on="nom_scientifique",
    how="left"
)

boite_df = boite_df.merge(
    souche_df[["id_souche", "code_souche", "id_espece"]],
    on=["code_souche", "id_espece"],
    how="left"
)

boite_df["id_boite"] = range(1, len(boite_df) + 1)
boite_df["global_id"] = "POLYPBASE-" + boite_df["id_boite"].astype(str).str.zfill(6)
boite_df["code_local"] = boite_df["code_boite"]
boite_df["date_creation"] = None
boite_df["statut_boite"] = "historique"

boite_df = boite_df[[
    "id_boite",
    "global_id",
    "code_local",
    "numero_boite_local",
    "date_creation",
    "statut_boite",
    "id_souche"
]]

print("Nombre de boîtes valides :", len(boite_df))
print("Doublons code_local :", boite_df["code_local"].duplicated().sum())
print("Boîtes sans souche :", boite_df["id_souche"].isna().sum())
display(boite_df.head(20))


## 10. Création des relevés biologiques

La table correspond à l'association `SAISIR_RELEVE` du MCD métier.


In [ ]:
releve_base_df = clean_tracking_valid_df.merge(
    boite_df[["id_boite", "code_local"]],
    left_on="code_boite",
    right_on="code_local",
    how="left"
)

print("Lignes de relevés sans boîte :", releve_base_df["id_boite"].isna().sum())

releve_df = (
    releve_base_df
    .pivot_table(
        index=["annee", "semaine", "id_boite"],
        columns="type_mesure",
        values="valeur",
        aggfunc="first"
    )
    .reset_index()
)

releve_df = releve_df.rename(columns={
    "polypes": "nombre_polypes",
    "ephyrules": "nombre_ephyrules"
})

releve_df["date_releve"] = (
    releve_df["annee"].astype(str)
    + "-S"
    + releve_df["semaine"].astype(int).astype(str).str.zfill(2)
)

releve_df["type_observation"] = "historique_excel"
releve_df["commentaire"] = None
releve_df["id_utilisateur"] = None

releve_df = releve_df[[
    "id_utilisateur",
    "id_boite",
    "date_releve",
    "nombre_polypes",
    "nombre_ephyrules",
    "type_observation",
    "commentaire"
]]

print("Nombre de relevés :", len(releve_df))
print("Relevés sans boîte :", releve_df["id_boite"].isna().sum())
print("Relevés sans polypes :", releve_df["nombre_polypes"].isna().sum())
print("Relevés sans éphyrules :", releve_df["nombre_ephyrules"].isna().sum())
display(releve_df.head(20))


In [ ]:
# Anomalie 4 : relevés incomplets.
anomalies_releves_incomplets_df = releve_df[
    releve_df["nombre_polypes"].isna() | releve_df["nombre_ephyrules"].isna()
].copy()

anomalies_releves_incomplets_df.to_csv(
    ANOMALY_DIR / "anomalies_releves_incomplets.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Relevés incomplets :", len(anomalies_releves_incomplets_df))
display(anomalies_releves_incomplets_df.head(20))


## 11. Création des zones thermiques et des rangements

Les zones sont créées à partir de la température cible observée dans les fichiers historiques.
Les lignes sans température sont conservées dans les relevés biologiques, mais ne sont pas utilisées pour créer `RANGE`.


In [ ]:
zone_thermique_df = (
    clean_tracking_valid_df[["temperature_cible"]]
    .dropna()
    .drop_duplicates()
    .sort_values("temperature_cible")
    .reset_index(drop=True)
)

zone_thermique_df["id_zone"] = range(1, len(zone_thermique_df) + 1)

zone_thermique_df["nom_zone"] = (
    "Zone "
    + zone_thermique_df["temperature_cible"].astype(str)
    + "°C"
)

zone_thermique_df["type_zone"] = "historique_excel"

zone_thermique_df = zone_thermique_df[[
    "id_zone",
    "nom_zone",
    "type_zone",
    "temperature_cible"
]]

print("Nombre de zones thermiques :", len(zone_thermique_df))
display(zone_thermique_df)
print(clean_tracking_valid_df["temperature_cible"].value_counts(dropna=False).sort_index())


In [ ]:
# Export des lignes valides biologiquement mais sans température cible.
anomalies_temperature_valid_df = clean_tracking_valid_df[
    clean_tracking_valid_df["temperature_cible"].isna()
].copy()

anomalies_temperature_valid_df.to_csv(
    ANOMALY_DIR / "anomalies_temperature_manquante_valides.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Lignes valides avec température manquante :", len(anomalies_temperature_valid_df))
display(anomalies_temperature_valid_df[[
    "code_boite",
    "espece",
    "annee",
    "semaine",
    "fichier_source",
    "feuille_source"
]].drop_duplicates().head(30))


In [ ]:
range_base_df = clean_tracking_valid_df[
    clean_tracking_valid_df["temperature_cible"].notna()
].copy()

range_base_df = range_base_df.merge(
    boite_df[["id_boite", "code_local"]],
    left_on="code_boite",
    right_on="code_local",
    how="left"
)

range_base_df = range_base_df.merge(
    zone_thermique_df[["id_zone", "temperature_cible"]],
    on="temperature_cible",
    how="left"
)

range_df = (
    range_base_df[[
        "id_boite",
        "id_zone",
        "annee",
        "semaine"
    ]]
    .dropna(subset=["id_boite", "id_zone"])
    .drop_duplicates()
    .reset_index(drop=True)
)

range_df["date_entree"] = (
    range_df["annee"].astype(str)
    + "-S"
    + range_df["semaine"].astype(int).astype(str).str.zfill(2)
)

range_df["date_sortie"] = None
range_df["motif"] = "historique_excel"
range_df["id_utilisateur"] = None

range_df = range_df[[
    "id_utilisateur",
    "id_boite",
    "id_zone",
    "date_entree",
    "date_sortie",
    "motif"
]]

print("Nombre de rangements :", len(range_df))
print("Rangements sans boîte :", range_df["id_boite"].isna().sum())
print("Rangements sans zone :", range_df["id_zone"].isna().sum())
display(range_df.head(20))


## 12. Exports finaux


In [ ]:
espece_df.to_csv(OUTPUT_DIR / "espece.csv", index=False, encoding="utf-8-sig")
souche_df.to_csv(OUTPUT_DIR / "souche.csv", index=False, encoding="utf-8-sig")
boite_df.to_csv(OUTPUT_DIR / "boite.csv", index=False, encoding="utf-8-sig")
releve_df.to_csv(OUTPUT_DIR / "saisir_releve.csv", index=False, encoding="utf-8-sig")
zone_thermique_df.to_csv(OUTPUT_DIR / "zone_thermique.csv", index=False, encoding="utf-8-sig")
range_df.to_csv(OUTPUT_DIR / "range.csv", index=False, encoding="utf-8-sig")

print("Exports terminés dans :", OUTPUT_DIR)
print("Anomalies exportées dans :", ANOMALY_DIR)


## 13. Résumé des contrôles

À vérifier après exécution :
- les boîtes valides ne doivent pas avoir de doublons ;
- les boîtes doivent toutes être reliées à une souche ;
- les relevés doivent tous être reliés à une boîte ;
- les rangements doivent tous être reliés à une boîte et à une zone ;
- les anomalies doivent être envoyées aux encadrants pour validation métier/scientifique.


In [ ]:
resume_controles = {
    "nb_especes": len(espece_df),
    "nb_souches": len(souche_df),
    "nb_boites_valides": len(boite_df),
    "nb_releves": len(releve_df),
    "nb_zones_thermiques": len(zone_thermique_df),
    "nb_rangements": len(range_df),
    "doublons_boites_valides": int(boite_df["code_local"].duplicated().sum()),
    "boites_sans_souche": int(boite_df["id_souche"].isna().sum()),
    "releves_sans_boite": int(releve_base_df["id_boite"].isna().sum()),
    "rangements_sans_boite": int(range_df["id_boite"].isna().sum()),
    "rangements_sans_zone": int(range_df["id_zone"].isna().sum()),
    "anomalies_temperature": len(anomalies_temperature_df),
    "anomalies_boites_plusieurs_especes": len(codes_boites_ambigues),
    "anomalies_releves_incomplets": len(anomalies_releves_incomplets_df),
}

resume_controles_df = pd.DataFrame(
    resume_controles.items(),
    columns=["controle", "valeur"]
)

display(resume_controles_df)
